# RAG with LangChain + OpenSearch (Complete Pipeline)

This notebook extends `RAG_Opensearch.ipynb` with a **full end-to-end RAG pipeline**:

```
Documents → Chunk → Embed → Store in OpenSearch
                                  ↓
Question → Embed → k-NN Search → Retrieve top-k chunks
                                  ↓
           Prompt = Question + Chunks → LLM → Answer
```

**What's new vs notebook 1:**
- Richer multi-document corpus with metadata (source, topic, date)
- Metadata filtering during retrieval
- `RetrievalQA` chain — single question answering
- `ConversationalRetrievalChain` — multi-turn chat with memory
- Source attribution — see which documents were used to answer
- Hybrid search (BM25 + vector)

**Prerequisites:**
```bash
# OpenSearch running locally
export OPENSEARCH_JAVA_HOME=$(/usr/libexec/java_home -v 21)
export OPENSEARCH_JAVA_OPTS="-Xms512m -Xmx512m"
./bin/opensearch

# Ollama models
ollama pull nomic-embed-text   # embeddings
ollama pull llama3             # LLM for generation

# Python packages
pip install langchain langchain-community langchain-ollama opensearch-py
```

## 1. Imports & Configuration

In [1]:
from langchain_community.vectorstores import OpenSearchVectorSearch
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain.chains import RetrievalQA, ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import warnings
warnings.filterwarnings('ignore')

# ── Connection config ──────────────────────────────────────────────────────────
OPENSEARCH_URL = "http://localhost:9200"
INDEX_NAME     = "rag-knowledge-base"
HTTP_AUTH      = ("admin", "admin")   # change if you configured custom credentials

# ── Models ────────────────────────────────────────────────────────────────────
EMBED_MODEL = "nomic-embed-text"  # 768-dim embeddings
LLM_MODEL   = "llama3"            # generation model

print("✅ Imports complete")

/Users/zainabfirdaus/git/airflow/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports complete


## 2. Knowledge Base — Multi-Document Corpus

A realistic corpus with documents from different topics and sources.  
Each document carries **metadata** (source, topic, date) that we can filter on during retrieval.

In [2]:
# Raw documents — simulating a company knowledge base
raw_docs = [
    {
        "text": """
        OpenSearch is a scalable, open-source search and analytics suite derived from Elasticsearch.
        It supports full-text search, structured search, analytics, and vector search (k-NN).
        OpenSearch includes OpenSearch Dashboards for data visualization.
        It is commonly used for log analytics, application monitoring, and as a vector database for RAG.
        The k-NN plugin supports HNSW and IVF algorithms via nmslib, faiss, and Lucene engines.
        """,
        "metadata": {"source": "opensearch_docs", "topic": "opensearch", "date": "2024-01"}
    },
    {
        "text": """
        RAG (Retrieval-Augmented Generation) is a technique that enhances LLM responses
        by first retrieving relevant documents from a knowledge base, then passing them
        as context to the language model along with the user's question.
        This reduces hallucinations and keeps answers grounded in real data.
        RAG consists of two phases: indexing (chunking + embedding + storing) and
        retrieval (embedding the query + k-NN search + LLM generation).
        """,
        "metadata": {"source": "ai_handbook", "topic": "rag", "date": "2024-02"}
    },
    {
        "text": """
        LangChain is a framework for building LLM-powered applications.
        Key components: Chains, Agents, Memory, Retrievers, Vector Stores, Prompts.
        LangChain Expression Language (LCEL) enables composing chains with the pipe operator.
        It integrates with OpenSearch, Chroma, Pinecone, Weaviate and many other vector stores.
        ConversationalRetrievalChain adds chat history awareness to standard RAG.
        """,
        "metadata": {"source": "langchain_docs", "topic": "langchain", "date": "2024-03"}
    },
    {
        "text": """
        Ollama is a tool to run large language models locally on your machine.
        It supports models like Llama 3, Mistral, Gemma, Phi-3, and many others.
        Models can be pulled with: ollama pull <model-name>
        Ollama exposes an OpenAI-compatible REST API at http://localhost:11434.
        It can be used for both embeddings (nomic-embed-text, mxbai-embed-large)
        and text generation (llama3, mistral, etc).
        """,
        "metadata": {"source": "ollama_docs", "topic": "ollama", "date": "2024-03"}
    },
    {
        "text": """
        Vector embeddings are numerical representations of text in high-dimensional space.
        Semantically similar texts have vectors that are close together (cosine similarity).
        Common embedding models: nomic-embed-text (768d), OpenAI text-embedding-3-small (1536d),
        sentence-transformers/all-MiniLM-L6-v2 (384d).
        k-NN (k-Nearest Neighbour) search finds the top-k most similar vectors to a query vector.
        HNSW (Hierarchical Navigable Small World) is the most popular approximate k-NN algorithm.
        """,
        "metadata": {"source": "ml_glossary", "topic": "embeddings", "date": "2024-01"}
    },
    {
        "text": """
        Chunking strategies for RAG:
        1. Fixed-size chunks: split every N characters, simple but may cut sentences.
        2. Recursive character splitter: tries to split on paragraphs, then sentences, then words.
        3. Semantic chunking: splits based on embedding similarity between sentences.
        4. Document-structure chunking: respects headers, sections (for Markdown/HTML).
        Chunk overlap (50-100 chars) preserves context across boundaries.
        Typical chunk sizes: 256–1024 tokens depending on document type.
        """,
        "metadata": {"source": "rag_best_practices", "topic": "rag", "date": "2024-04"}
    },
]

print(f"Loaded {len(raw_docs)} source documents")

Loaded 6 source documents


## 3. Chunk the Documents

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " ", ""]
)

all_chunks = []
for doc in raw_docs:
    chunks = splitter.split_text(doc["text"].strip())
    for chunk in chunks:
        all_chunks.append(
            Document(
                page_content=chunk,
                metadata=doc["metadata"]    # preserve source/topic/date on every chunk
            )
        )

print(f"Total chunks after splitting: {len(all_chunks)}")
print(f"\nSample chunk:")
print(f"  Content: {all_chunks[0].page_content[:120]}...")
print(f"  Metadata: {all_chunks[0].metadata}")

Total chunks after splitting: 12

Sample chunk:
  Content: OpenSearch is a scalable, open-source search and analytics suite derived from Elasticsearch.
        It supports full-te...
  Metadata: {'source': 'opensearch_docs', 'topic': 'opensearch', 'date': '2024-01'}


## 4. Embeddings + Index into OpenSearch

In [4]:
# Initialize the embedding model (runs locally via Ollama)
embedding_model = OllamaEmbeddings(model=EMBED_MODEL)

# Test the embedding model works before indexing
test_embed = embedding_model.embed_query("test")
print(f"✅ Embedding model OK — vector dimension: {len(test_embed)}")

# Index all chunks into OpenSearch
# This creates the index with knn_vector mapping and inserts all documents
vectorstore = OpenSearchVectorSearch.from_documents(
    documents=all_chunks,
    embedding=embedding_model,
    opensearch_url=OPENSEARCH_URL,
    index_name=INDEX_NAME,
    http_auth=HTTP_AUTH,
    use_ssl=False,
    verify_certs=False,
    engine="nmslib",           # nmslib, faiss, or lucene
    space_type="cosinesimil",  # cosinesimil, l2, innerproduct
    bulk_size=500,
)

print(f"✅ Indexed {len(all_chunks)} chunks into OpenSearch index '{INDEX_NAME}'")

✅ Embedding model OK — vector dimension: 768


RequestError: RequestError(400, 'settings_exception', 'unknown setting [index.knn] please check that any required plugins are installed, or check the breaking changes documentation for removed settings')

## 5. Connect to Existing Index (use this if index is already created)

In [ ]:
# Use this cell instead of cell 4 if you've already indexed documents
# and just want to query the existing index

vectorstore = OpenSearchVectorSearch(
    index_name=INDEX_NAME,
    embedding_function=embedding_model,
    opensearch_url=OPENSEARCH_URL,
    http_auth=HTTP_AUTH,
    use_ssl=False,
    verify_certs=False,
)

print(f"✅ Connected to existing index '{INDEX_NAME}'")

## 6. Retrieval — Similarity Search

Before wiring up the full RAG chain, test that retrieval works correctly.

In [ ]:
query = "How does RAG reduce hallucinations?"

# Basic k-NN similarity search — returns top 3 most relevant chunks
results = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}")
print(f"Top {len(results)} retrieved chunks:\n")
for i, doc in enumerate(results):
    print(f"[{i+1}] Source: {doc.metadata['source']} | Topic: {doc.metadata['topic']}")
    print(f"    {doc.page_content[:200]}")
    print()

In [ ]:
# Similarity search WITH relevance scores
results_with_scores = vectorstore.similarity_search_with_relevance_scores(query, k=3)

print(f"Query: {query}\n")
for doc, score in results_with_scores:
    print(f"Score: {score:.4f} | Source: {doc.metadata['source']}")
    print(f"  {doc.page_content[:150]}")
    print()

## 7. Metadata Filtering

Restrict retrieval to documents matching specific metadata — e.g. only return chunks from the `rag` topic.

In [ ]:
# Only retrieve chunks where topic == "rag"
filtered_results = vectorstore.similarity_search(
    query="What is chunking?",
    k=3,
    filter=[{"term": {"metadata.topic": "rag"}}]   # OpenSearch DSL filter
)

print("Filtered search (topic=rag only):")
for doc in filtered_results:
    print(f"  [{doc.metadata['source']}] {doc.page_content[:150]}")
    print()

## 8. RetrievalQA Chain — Single Question Answering

The standard RAG chain:
1. Embed the question
2. Retrieve top-k chunks from OpenSearch
3. Build a prompt: `context (chunks) + question`
4. Pass to the LLM
5. Return the answer + source documents

In [ ]:
# Custom prompt template — tells the LLM exactly how to use the context
QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a helpful assistant. Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't have enough information to answer that."
Do not make up information.

Context:
{context}

Question: {question}

Answer:"""
)

# LLM — local Ollama
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0,       # 0 = deterministic, good for factual Q&A
)

# Retriever — wraps the vectorstore with a k parameter
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}    # retrieve top 4 chunks
)

# Build the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",           # stuff = concatenate all chunks into one prompt
    retriever=retriever,
    return_source_documents=True, # include which chunks were used
    chain_type_kwargs={"prompt": QA_PROMPT}
)

print("✅ RetrievalQA chain ready")

In [ ]:
def ask(question):
    """Ask a question and print the answer with sources."""
    print(f"❓ Question: {question}\n")
    
    result = qa_chain.invoke({"query": question})
    
    print(f"💬 Answer:\n{result['result']}\n")
    print("📚 Sources used:")
    seen = set()
    for doc in result["source_documents"]:
        src = doc.metadata.get("source", "unknown")
        if src not in seen:
            print(f"  - {src} (topic: {doc.metadata.get('topic', 'N/A')})")
            seen.add(src)
    print("-" * 60)

# Test with several questions
ask("What is RAG and how does it reduce hallucinations?")

In [ ]:
ask("What embedding models does Ollama support?")

In [ ]:
ask("What chunking strategies are recommended for RAG?")

In [ ]:
# This should return "I don't have enough information" — not in the corpus
ask("What is the capital of France?")

## 9. LCEL Chain (LangChain Expression Language)

Modern LangChain way to build the same RAG pipeline using the pipe operator `|`.
More flexible and composable than `RetrievalQA`.

In [ ]:
def format_docs(docs):
    """Concatenate retrieved chunks into a single context string."""
    return "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in docs
    )

# LCEL prompt
lcel_prompt = ChatPromptTemplate.from_template("""
You are a knowledgeable assistant. Answer based ONLY on the provided context.
If unsure, say so.

Context:
{context}

Question: {question}

Answer:
""")

# LCEL RAG chain — reads left to right
# 1. RunnablePassthrough passes the question through unchanged
# 2. retriever fetches relevant docs
# 3. format_docs formats them
# 4. lcel_prompt builds the full prompt
# 5. llm generates the answer
# 6. StrOutputParser extracts the text
lcel_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | lcel_prompt
    | llm
    | StrOutputParser()
)

# Invoke
question = "How does OpenSearch support vector search?"
print(f"❓ {question}\n")
answer = lcel_chain.invoke(question)
print(f"💬 {answer}")

## 10. Conversational RAG — Multi-turn Chat with Memory

`ConversationalRetrievalChain` maintains chat history so follow-up questions
like *"Tell me more about that"* or *"What did you mean by HNSW?"* work correctly.

In [ ]:
# Memory stores the conversation history
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

# Conversational RAG chain
conv_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    verbose=False
)

def chat(question):
    """Send a message in the conversation."""
    print(f"👤 You: {question}")
    result = conv_chain.invoke({"question": question})
    print(f"🤖 Bot: {result['answer']}")
    print(f"   (used {len(result['source_documents'])} chunks from: "
          f"{set(d.metadata['source'] for d in result['source_documents'])})")
    print()

print("✅ Conversational chain ready. Starting conversation...\n")

chat("What is LangChain?")

In [ ]:
# Follow-up — relies on chat history to understand "it"
chat("What vector stores does it integrate with?")

In [ ]:
chat("How is that different from a regular search engine like plain OpenSearch?")

In [ ]:
# Inspect memory — see the full conversation history
print("📋 Full conversation history:")
for msg in memory.chat_memory.messages:
    role = "👤 Human" if msg.type == "human" else "🤖 AI"
    print(f"\n{role}: {msg.content[:300]}")

## 11. Verify Index in OpenSearch

Useful curl commands to inspect what was indexed.

In [ ]:
import requests

def opensearch_get(path):
    resp = requests.get(
        f"{OPENSEARCH_URL}{path}",
        auth=HTTP_AUTH,
        verify=False
    )
    return resp.json()

# Index stats
stats = opensearch_get(f"/{INDEX_NAME}/_count")
print(f"Documents in index '{INDEX_NAME}': {stats.get('count', 'N/A')}")

# Index mapping — shows the knn_vector field
mapping = opensearch_get(f"/{INDEX_NAME}/_mapping")
props = mapping.get(INDEX_NAME, {}).get('mappings', {}).get('properties', {})
print(f"\nIndex fields:")
for field, config in props.items():
    print(f"  {field}: {config.get('type', 'object')}")

## 12. Summary — RAG Pipeline Flow

```
INDEXING PHASE (run once)
─────────────────────────
Raw Text
  └─► RecursiveCharacterTextSplitter  → chunks (400 chars, 60 overlap)
        └─► OllamaEmbeddings           → 768-dim vectors per chunk
              └─► OpenSearchVectorSearch.from_documents()
                    └─► OpenSearch knn_vector index (nmslib / cosinesimil)

RETRIEVAL + GENERATION PHASE (per query)
─────────────────────────────────────────
User Question
  └─► OllamaEmbeddings (embed the question)
        └─► OpenSearch k-NN search    → top-k relevant chunks
              └─► PromptTemplate       → "Context: {chunks}\nQuestion: {q}"
                    └─► ChatOllama (llama3)
                          └─► Final Answer (grounded in retrieved docs)

CONVERSATIONAL LAYER (optional)
────────────────────────────────
ConversationBufferMemory stores [Human, AI, Human, AI, ...]
ConversationalRetrievalChain rewrites follow-up questions using history
before sending to the retriever.
```

**Key parameters to tune:**
| Parameter | Effect |
|---|---|
| `chunk_size` | Larger = more context per chunk, fewer chunks retrieved |
| `chunk_overlap` | Higher = less information loss at boundaries |
| `k` in retriever | More chunks = more context but longer prompt |
| `temperature` | 0 = factual, 0.7+ = creative |
| `space_type` | `cosinesimil` for text, `l2` for image embeddings |